# Shapessocial Structure Comparison Results

This notebook reads the completed `remote_experiment_l8_l13_l22_full_real` analysis and gives us a first interactive pass over:

- family-level brain and LM metrics
- top brain parcels and LM targets
- shared feature importance
- parcel-to-LM similarity structure
- basic diagnostics for NaN LM targets

It is designed to run in the current local environment without extra plotting libraries.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'structure_comparison').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing structure_comparison/')


ROOT = find_repo_root()
RUN_DIR = ROOT / 'structure_comparison' / 'outputs' / 'remote_experiment_l8_l13_l22_full_real'
BRAIN_TARGETS = ROOT / 'structure_comparison' / 'brain_targets' / 'shapessocial_schaefer200_full.npz'
BRAIN_SUMMARY = ROOT / 'structure_comparison' / 'brain_targets' / 'shapessocial_schaefer200_full.summary.json'
FAMILIES = ['layer8', 'layer13', 'layer22', 'all_layers']

print(f'ROOT: {ROOT}')
print(f'RUN_DIR: {RUN_DIR}')
print(f'BRAIN_TARGETS: {BRAIN_TARGETS}')


In [ ]:
def load_json(path: Path):
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)


analysis_summary = load_json(RUN_DIR / 'analysis_summary.json')
brain_summary = load_json(BRAIN_SUMMARY)

brain_npz = np.load(BRAIN_TARGETS, allow_pickle=True)
print('Brain matrix shape:', brain_npz['values'].shape)
print('Parcel count:', len(brain_npz['target_names']))
print('Resolved runs:', brain_summary['resolved_bold_runs'])
print('Expected TRs per run:', brain_summary['expected_trs_per_run'])
print('Total samples:', brain_summary['total_samples'])


In [ ]:
overview_rows = []
family_summaries = {}

for family in FAMILIES:
    summary = load_json(RUN_DIR / family / 'summary.json')
    family_summaries[family] = summary
    overview_rows.append(
        {
            'family': family,
            'predictors': summary['predictor_count'],
            'lm_targets': summary['lm_target_count'],
            'brain_mean_r': summary['brain_mean_test_correlation'],
            'brain_mean_r2': summary['brain_mean_test_r2'],
            'lm_mean_r': summary['lm_mean_test_correlation'],
            'lm_mean_r2': summary['lm_mean_test_r2'],
            'sample_rsa': summary['sample_rsa_correlation'],
            'feature_importance_r': summary['feature_importance_correlation'],
            'brain_alpha': summary['brain_consensus_alpha'],
            'lm_alpha': summary['lm_consensus_alpha'],
        }
    )

overview = pd.DataFrame(overview_rows).sort_values('family').reset_index(drop=True)
display(overview)


In [ ]:
def make_bar(value: float, width: int = 16) -> str:
    if pd.isna(value):
        return ''
    value = max(min(float(value), 1.0), -1.0)
    filled = int(round(abs(value) * width))
    block = '#' * filled + '.' * (width - filled)
    return ('-' if value < 0 else '+') + block


parcel_rows = []
lm_rows = []

for family, summary in family_summaries.items():
    for rank, row in enumerate(summary['top_parcels'], start=1):
        parcel_rows.append(
            {
                'family': family,
                'rank': rank,
                'parcel': row['target_name'],
                'corr': row['correlation'],
                'r2': row['r2'],
                'bar': make_bar(row['correlation']),
            }
        )
    for rank, row in enumerate(summary['top_lm_targets'], start=1):
        lm_rows.append(
            {
                'family': family,
                'rank': rank,
                'lm_target': row['target_name'],
                'corr': row['correlation'],
                'r2': row['r2'],
                'bar': make_bar(row['correlation']),
            }
        )

top_parcels = pd.DataFrame(parcel_rows)
top_lm_targets = pd.DataFrame(lm_rows)

display(top_parcels.groupby('family').head(5))
display(top_lm_targets.groupby('family').head(5))


In [ ]:
feature_importance = load_json(RUN_DIR / 'all_layers' / 'feature_importance_summary.json')
top_features = pd.DataFrame(feature_importance['top_features'])
top_features['brain_rank'] = top_features['brain_importance'].rank(ascending=False, method='dense').astype(int)
top_features['lm_rank'] = top_features['lm_importance'].rank(ascending=False, method='dense').astype(int)

print('All-layers feature importance correlation:', feature_importance['importance_correlation'])
display(top_features.head(20))


In [ ]:
sim_npz = np.load(RUN_DIR / 'all_layers' / 'brain_lm_weight_similarity.npz', allow_pickle=True)
sim = sim_npz['similarity']
brain_names = sim_npz['brain_target_names']
lm_names = sim_npz['lm_target_names']

pairs = []
for i, brain_name in enumerate(brain_names):
    for j, lm_name in enumerate(lm_names):
        pairs.append(
            {
                'parcel': brain_name,
                'lm_target': lm_name,
                'similarity': float(sim[i, j]),
            }
        )

similarity_df = pd.DataFrame(pairs).sort_values('similarity', ascending=False).reset_index(drop=True)
display(similarity_df.head(25))


In [ ]:
lm_diag_rows = []
for family in FAMILIES:
    lm_cv = load_json(RUN_DIR / family / 'lm_cv_summary.json')
    lm_final = np.load(RUN_DIR / family / 'lm_final_model.npz', allow_pickle=True)
    target_names = list(lm_final['target_names'])
    per_target_corr = lm_cv['aggregate']['per_target_mean_correlation']
    per_target_r2 = lm_cv['aggregate']['per_target_mean_r2']
    for target_name, corr, r2 in zip(target_names, per_target_corr, per_target_r2):
        lm_diag_rows.append(
            {
                'family': family,
                'lm_target': target_name,
                'corr': corr,
                'r2': r2,
                'is_nan': pd.isna(corr) or pd.isna(r2),
            }
        )

lm_diag = pd.DataFrame(lm_diag_rows)
nan_counts = lm_diag.groupby('family')['is_nan'].sum().rename('nan_target_count').reset_index()
display(nan_counts)
display(lm_diag[lm_diag['is_nan']].sort_values(['family', 'lm_target']))


In [ ]:
brain_model = np.load(RUN_DIR / 'all_layers' / 'brain_final_model.npz', allow_pickle=True)
collapsed_weights = brain_model['collapsed_weights']
feature_names = brain_model['base_feature_names']
parcel_names = brain_model['target_names']

feature_strength = pd.DataFrame(
    {
        'feature_name': feature_names,
        'mean_abs_weight': np.mean(np.abs(collapsed_weights), axis=1),
        'max_abs_weight': np.max(np.abs(collapsed_weights), axis=1),
    }
).sort_values('mean_abs_weight', ascending=False)

parcel_strength = pd.DataFrame(
    {
        'parcel': parcel_names,
        'mean_abs_weight': np.mean(np.abs(collapsed_weights), axis=0),
        'max_abs_weight': np.max(np.abs(collapsed_weights), axis=0),
    }
).sort_values('mean_abs_weight', ascending=False)

display(feature_strength.head(20))
display(parcel_strength.head(20))


## Quick Read

- `all_layers` currently gives the strongest mean held-out brain correlation of the four families.
- Brain-side `R^2` values are still very negative, so this is not yet a well-calibrated encoding model in variance-explained terms.
- The feature-importance agreement is strongest for `layer8` and `all_layers`, weaker for `layer13`.
- Some LM targets return `NaN` metrics, which is worth checking before we treat the LM-side summary as stable.
- The top brain parcels are concentrated in visual and dorsal attention regions, which is a useful sanity-check but also something to interrogate carefully.

Next notebook steps could be: better residual diagnostics, per-run breakdowns, ROI grouping, and direct inspection of the top shared features from the VM concept annotations.